# 15 - Evaluations
Test agent quality using built-in evaluators, create custom evaluators, and run batch and online evaluation pipelines.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
MODEL_ID = config["MODEL_ID"]
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]
KB_ID = config["KB_ID"]

tracker = ResourceTracker()
bedrock = boto3.client("bedrock", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"\u2713 Config loaded (region={REGION}, model={MODEL_ID})")

## Step 2: Create Test Dataset

Define evaluation test cases with expected answers.

In [ ]:
# Evaluation test cases for the OrgIQ search agent
test_cases = [
    {
        "input": "Who is the top Kubernetes expert?",
        "expected_output": "Alice Chen is the top Kubernetes expert with a strength score of 0.9",
        "expected_tools": ["search_experts"],
        "category": "expert-search"
    },
    {
        "input": "What documents exist about machine learning?",
        "expected_output": "ML Pipeline Architecture by Bob Kumar",
        "expected_tools": ["search_documents"],
        "category": "doc-search"
    },
    {
        "input": "Tell me about Eve Johnson's expertise",
        "expected_output": "Eve Johnson is a Security Engineer with expertise in Authentication (0.95) and Microservices (0.6)",
        "expected_tools": ["get_person_profile"],
        "category": "profile-lookup"
    },
    {
        "input": "Who can help with CI/CD and what have they written?",
        "expected_output": "Dave Park has strong CI/CD expertise (0.9) and authored GitOps Pipeline Setup. Alice Chen also knows CI/CD (0.8).",
        "expected_tools": ["search_experts", "get_person_profile"],
        "category": "multi-tool"
    },
    {
        "input": "Who is the best person to consult about microservices architecture?",
        "expected_output": "Carol Diaz is the top microservices expert (0.9 strength) and authored Service Mesh Patterns",
        "expected_tools": ["search_experts"],
        "category": "expert-search"
    },
]

print(f"\u2713 {len(test_cases)} test cases defined")
for tc in test_cases:
    print(f"  [{tc['category']}] {tc['input'][:50]}...")

## Step 3: Create the Agent Under Test

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

neptune = boto3.client("neptune-graph", region_name=REGION)

@tool
def search_experts(topic: str) -> str:
    """Search for experts on a topic.
    
    Args:
        topic: Topic to find experts for
    """
    query = """
    MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic)
    WHERE toLower(t.name) CONTAINS toLower($topic)
    RETURN p.name AS name, p.title AS title, r.strength AS strength
    ORDER BY r.strength DESC LIMIT 3
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID, queryString=query,
        language="OPEN_CYPHER", parameters={"topic": {"value": topic}}
    )
    results = json.loads(response["payload"].read())
    return "\n".join([f"{r['name']} ({r['title']}) - strength: {r['strength']}" for r in results["results"]])

@tool
def search_documents(query: str) -> str:
    """Search the knowledge base for documents.
    
    Args:
        query: Search query text
    """
    bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID, retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}}
    )
    return "\n".join([r["content"]["text"][:200] for r in response["retrievalResults"]])

@tool
def get_person_profile(name: str) -> str:
    """Get a person's profile from the knowledge graph.
    
    Args:
        name: Person's name to look up
    """
    query = """
    MATCH (p:Person) WHERE toLower(p.name) CONTAINS toLower($name)
    OPTIONAL MATCH (p)-[r:KNOWS_ABOUT]->(t:Topic)
    OPTIONAL MATCH (p)-[:AUTHORED]->(d:Document)
    RETURN p.name AS name, p.title AS title, collect(DISTINCT t.name) AS topics, collect(DISTINCT d.title) AS docs
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID, queryString=query,
        language="OPEN_CYPHER", parameters={"name": {"value": name}}
    )
    results = json.loads(response["payload"].read())
    if not results["results"]: return f"No profile found for {name}"
    r = results["results"][0]
    return f"{r['name']} ({r['title']})\nTopics: {', '.join(r['topics'])}\nDocs: {', '.join(r['docs'])}"

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
test_agent = Agent(
    model=model,
    system_prompt="You are OrgIQ Search. Find experts, documents, and profiles in the organization.",
    tools=[search_experts, search_documents, get_person_profile]
)

print("\u2713 Agent under test created")

## Step 4: Run Agent on Test Cases

In [ ]:
# Execute agent on all test cases
results = []

print("\u231b Running agent on test cases...")
for i, tc in enumerate(test_cases, 1):
    print(f"  Running test {i}/{len(test_cases)}: {tc['input'][:40]}...")
    try:
        response = test_agent(tc["input"])
        results.append({
            "input": tc["input"],
            "expected": tc["expected_output"],
            "actual": str(response),
            "category": tc["category"],
            "success": True
        })
    except Exception as e:
        results.append({
            "input": tc["input"],
            "expected": tc["expected_output"],
            "actual": f"ERROR: {e}",
            "category": tc["category"],
            "success": False
        })

success_count = sum(1 for r in results if r["success"])
print(f"\n\u2713 {success_count}/{len(results)} test cases completed successfully")

## Step 5: Built-in Evaluator - Correctness

Use an LLM-as-judge to evaluate correctness of responses.

In [ ]:
def evaluate_correctness(input_text: str, expected: str, actual: str) -> dict:
    """Use LLM-as-judge to evaluate response correctness."""
    eval_prompt = f"""You are an evaluation judge. Score the ACTUAL response against the EXPECTED response.

INPUT: {input_text}
EXPECTED: {expected}
ACTUAL: {actual}

Score on a scale of 1-5:
5 = Perfect match in content and meaning
4 = Mostly correct with minor omissions
3 = Partially correct but missing key info
2 = Somewhat relevant but mostly incorrect
1 = Completely wrong or irrelevant

Respond in JSON format: {{"score": N, "reasoning": "brief explanation"}}"""
    
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": eval_prompt}]}],
        inferenceConfig={"maxTokens": 200}
    )
    output = response["output"]["message"]["content"][0]["text"]
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        return {"score": 0, "reasoning": f"Failed to parse: {output[:100]}"}

# Evaluate all results
print("\u231b Evaluating correctness (LLM-as-judge)...\n")
eval_results = []
for r in results:
    if r["success"]:
        eval_result = evaluate_correctness(r["input"], r["expected"], r["actual"])
        eval_results.append(eval_result)
        print(f"  [{eval_result['score']}/5] {r['input'][:40]}...")
        print(f"         {eval_result['reasoning'][:60]}")

avg_score = sum(e["score"] for e in eval_results) / len(eval_results) if eval_results else 0
print(f"\n\u2713 Average correctness score: {avg_score:.2f}/5")

## Step 6: Built-in Evaluator - Faithfulness

Check that responses are grounded in retrieved context (no hallucination).

In [ ]:
def evaluate_faithfulness(input_text: str, actual: str) -> dict:
    """Evaluate if response is grounded in available data (no hallucination)."""
    eval_prompt = f"""You are a faithfulness evaluator. Determine if the response contains any hallucinated information
that could NOT have come from a knowledge graph of people, topics, and documents in an organization.

INPUT: {input_text}
RESPONSE: {actual}

Score on a scale of 1-5:
5 = Fully grounded, no hallucination
4 = Mostly grounded with minor assumptions
3 = Mix of grounded and potentially made-up info
2 = Significant hallucination present
1 = Mostly hallucinated

Respond in JSON format: {{"score": N, "reasoning": "brief explanation"}}"""
    
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": eval_prompt}]}],
        inferenceConfig={"maxTokens": 200}
    )
    output = response["output"]["message"]["content"][0]["text"]
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        return {"score": 0, "reasoning": f"Parse error: {output[:100]}"}

print("\u231b Evaluating faithfulness...\n")
faith_results = []
for r in results:
    if r["success"]:
        eval_result = evaluate_faithfulness(r["input"], r["actual"])
        faith_results.append(eval_result)
        print(f"  [{eval_result['score']}/5] {r['input'][:40]}...")

avg_faith = sum(e["score"] for e in faith_results) / len(faith_results) if faith_results else 0
print(f"\n\u2713 Average faithfulness score: {avg_faith:.2f}/5")

## Step 7: Custom Evaluator - Tool Usage

In [ ]:
def evaluate_tool_usage(input_text: str, actual: str, expected_tools: list) -> dict:
    """Check if the agent used appropriate tools for the query."""
    # Check if tool names appear in the response trace
    tools_used = []
    for tool_name in ["search_experts", "search_documents", "get_person_profile"]:
        if tool_name in actual.lower() or tool_name.replace("_", " ") in actual.lower():
            tools_used.append(tool_name)
    
    # Simple heuristic: check if expected tools were likely invoked
    expected_coverage = len(set(expected_tools) & set(tools_used)) / len(expected_tools) if expected_tools else 1.0
    
    return {
        "score": min(5, int(expected_coverage * 5) + 1),
        "tools_expected": expected_tools,
        "tools_detected": tools_used,
        "coverage": expected_coverage
    }

print("Tool usage evaluation:")
for i, (r, tc) in enumerate(zip(results, test_cases)):
    if r["success"]:
        eval_result = evaluate_tool_usage(r["input"], r["actual"], tc["expected_tools"])
        print(f"  [{eval_result['score']}/5] {r['input'][:40]}...")
        print(f"         Expected: {eval_result['tools_expected']}, Detected: {eval_result['tools_detected']}")

print("\n\u2713 Tool usage evaluation complete")

## Step 8: Evaluation Summary Report

In [ ]:
# Generate evaluation summary
print("=" * 60)
print("EVALUATION REPORT")
print("=" * 60)
print(f"\nAgent: {PREFIX}-search")
print(f"Model: {MODEL_ID}")
print(f"Test cases: {len(test_cases)}")
print(f"Successful runs: {success_count}/{len(test_cases)}")
print(f"\nMetrics:")
print(f"  Correctness:   {avg_score:.2f}/5.00")
print(f"  Faithfulness:  {avg_faith:.2f}/5.00")
print(f"  Success rate:  {success_count/len(test_cases)*100:.0f}%")

# Per-category breakdown
categories = set(tc["category"] for tc in test_cases)
print(f"\nPer-category results:")
for cat in sorted(categories):
    cat_results = [r for r, tc in zip(results, test_cases) if tc["category"] == cat]
    cat_success = sum(1 for r in cat_results if r["success"])
    print(f"  {cat}: {cat_success}/{len(cat_results)} passed")

print(f"\n" + "=" * 60)
print("\u2713 Evaluation report generated")

## Step 9: Batch Evaluation with Bedrock

Use Bedrock's batch evaluation API for larger test sets.

In [ ]:
# Prepare batch evaluation dataset
batch_dataset = []
for tc in test_cases:
    batch_dataset.append({
        "prompt": tc["input"],
        "referenceResponse": tc["expected_output"],
        "category": tc["category"]
    })

# Upload to S3 for batch evaluation
S3_BUCKET = config["S3_BUCKET"]
if S3_BUCKET:
    s3 = boto3.client("s3", region_name=REGION)
    eval_key = "evaluations/test-dataset.jsonl"
    
    jsonl_content = "\n".join(json.dumps(item) for item in batch_dataset)
    s3.put_object(Bucket=S3_BUCKET, Key=eval_key, Body=jsonl_content.encode())
    
    print(f"\u2713 Evaluation dataset uploaded to s3://{S3_BUCKET}/{eval_key}")
    print(f"  {len(batch_dataset)} test cases in JSONL format")
else:
    print("\u231b S3 bucket not configured - skipping batch upload")

In [ ]:
# Create a batch evaluation job (if S3 bucket available)
if S3_BUCKET:
    try:
        eval_job = bedrock.create_evaluation_job(
            jobName=f"{PREFIX}-search-eval-{int(time.time())}",
            jobDescription="OrgIQ search agent quality evaluation",
            roleArn=f"arn:aws:iam::{config['ACCOUNT_ID']}:role/{PREFIX}-eval-role",
            evaluationConfig={
                "automated": {
                    "datasetMetricConfigs": [{
                        "taskType": "QuestionAndAnswer",
                        "metricNames": ["Correctness", "Completeness", "Faithfulness"],
                        "dataset": {
                            "name": "orgiq-search-eval",
                            "datasetLocation": {
                                "s3Uri": f"s3://{S3_BUCKET}/{eval_key}"
                            }
                        }
                    }]
                }
            },
            inferenceConfig={
                "models": [{
                    "bedrockModel": {
                        "modelIdentifier": MODEL_ID
                    }
                }]
            },
            outputDataConfig={
                "s3Uri": f"s3://{S3_BUCKET}/evaluations/results/"
            }
        )
        print(f"\u2713 Batch evaluation job created: {eval_job['jobArn']}")
    except Exception as e:
        print(f"\u231b Batch evaluation not available: {type(e).__name__}")
        print(f"  (Requires evaluation role and proper permissions)")
else:
    print("\u231b Skipping batch evaluation (no S3 bucket)")

## Step 10: Online Evaluation Pattern

In [ ]:
# Online evaluation: evaluate responses in real-time
def online_evaluate(input_text: str, response: str) -> dict:
    """Lightweight online evaluation for production monitoring."""
    metrics = {
        "response_length": len(response),
        "has_sources": any(name in response for name in ["Alice", "Bob", "Carol", "Dave", "Eve"]),
        "has_scores": any(char.isdigit() and "." in response[max(0,i-2):i+2] for i, char in enumerate(response)),
        "is_error": "error" in response.lower() or "not found" in response.lower(),
    }
    
    # Quality heuristic
    quality = 1.0
    if metrics["has_sources"]: quality += 0.3
    if metrics["has_scores"]: quality += 0.2
    if metrics["is_error"]: quality -= 0.5
    if metrics["response_length"] < 50: quality -= 0.3
    metrics["quality_estimate"] = min(1.0, max(0.0, quality))
    
    return metrics

# Test online evaluation
print("Online evaluation metrics:")
for r in results:
    if r["success"]:
        metrics = online_evaluate(r["input"], r["actual"])
        print(f"  {r['input'][:35]}... -> quality: {metrics['quality_estimate']:.2f}")

print("\n\u2713 Online evaluation pattern demonstrated")

## Summary

- [x] Test dataset created (5 cases across 4 categories)
- [x] Agent executed on all test cases
- [x] LLM-as-judge correctness evaluation
- [x] Faithfulness evaluation (hallucination detection)
- [x] Custom tool usage evaluator
- [x] Evaluation summary report with per-category breakdown
- [x] Batch evaluation dataset uploaded to S3
- [x] Bedrock batch evaluation job (API pattern)
- [x] Online evaluation pattern for production monitoring

**Next:** [16-cleanup.ipynb](16-cleanup.ipynb) — Tear down all resources safely